特征缩放(Feature Scaling)
概念：是机器学习中的一种数据预处理技术，旨在将不同量纲（单位）或者是数据范围内的原始数据调整到统一的尺度上，也就是说，就是把所有特征的数值范围强行按比例拉平

比如：
假设我们要让机器学习模型来判断两个人的体型是不是差不多
模型只能看到两个数据（特征）：
1. 身高（单位是m）：比如1.70m
2. 体重（单位是g）：比如70000g（也就是70kg）
现在有三个人
小明：身高1.70m，体重70000g（70kg）
小红：身高1.80m，体重70000g（70kg）
小刚：身高1.70m，体重70100g（70.1kg）
不难看出，小明和小刚的体型几乎是一模一样的，而小红比他们高很多，体型差异很大
但是，模型在训练的时候，会认为：
小明和小红的身高差了0.1，体重差了0。总差异算作0.1
小明和小刚的身高差了0，体重差了100，总差异算作100
于是，模型会得出这样一个结论：小明和小红的体型差距非常小，小明和小航的体型差距非常大，这种结论很显然是错的
为了解决这个问题，我们得要用到一个数据预处理技术，也就是特征缩放

在scikit-learn中，特征缩放有三种方式：
标准化(StandardScaler)
归一化(MinMaxScaler)
正则化(Normalizer)
上述三种class都是在sklearn.preprocessing这个工具箱里面
下面先讲标准化

标准化的计算方法是：算出所有数据的平均值，然后再计算标准差（用每一个原始数据减去平均值，然后将这个差值平方，将所有差值的平方加起来，再除以总个数，得到方差，再对方差进行开平方），最后带入公式：新数值 = （原始数据-平均值）/ 标准差

In [ ]:
from sklearn.preprocessing import StandardScaler # StandardScaler 是一个类

# 1. 召唤“公平秤”工具
scaler = StandardScaler()

# 2. 对“复习资料”（训练集）进行缩放
# fit_transform 的意思是：先去计算训练集的平均值和标准差(fit)，然后用刚才算出来的平均值作为标准，把大家的数据都去按照比例放缩(transform)
X_train_scaled = scaler.fit_transform(X_train)

# 3. 对“考卷”（测试集）进行缩放
# 注意这里只有 transform。因为模型是根据在平时的学习训练中，按照在训练集的标准来找规律的，如果在测试集中又新算出平均值，容易乱套（比例对不上），那就体现不出模型在平时训练中的表现了
# 要牢记，机器学习的本质就是在已知中训练，去解决未知的问题
X_test_scaled = scaler.transform(X_test)

将这个文件中涉及到的代码知识点和supervised_learning.ipynb中的结合起来，就能够得出一套完整的模型训练流水线

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier # 详情请看Ensemble_Learning.ipynb文件
from sklearn.metrics import accuracy_score

# 特征X：重量（kg），跑分，续航（h）
X = np.array([
    [3.5, 15000, 2.0],  # 游戏本
    [1.2,  4000, 10.0], # 轻薄本
    [2.8, 12000, 3.5],  # 全能本
    [3.2, 16000, 1.5],  # 游戏本
    [1.0,  3500, 12.0]  # 轻薄本
])

# 标签 y：1（是游戏本），0（不是）
y = np.array([1, 0, 0, 1, 0])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(random_state=42)

# model.fit() 的意思是：使用训练集的特征和标签来拟合模型，总结公式
model.fit(X_train_scaled, y_train)

# model.predict() 的意思是：使用训练好的模型，对测试集特征进行预测，并将结果存入变量中
# 返回的结果是一个numpy数组(np.ndarray)，就跟普通列表一样（使用方法），但是计算更快
predictions = model.predict(X_test_scaled)

print(f"模型的预测答案：{predictions}")
print(f"真实的正确答案：{y_test}")
print(f"模型准确率：{accuracy_score(y_test, predictions) * 100}%")

模型的预测答案：[0]
真实的正确答案：[0]
模型准确率：100.0%


归一化的计算方法是：把所有的数据，强行按比例挤压到一个固定的箱子里，通常是0-1
比如：
假设一次考试，满分是150分，全班最高考了150分，最低考了50分
考 50 分的人，进度条是 0%（对应数字 0）
考 150 分的人，进度条是 100%（对应数字 1）
考 100 分的人，刚好在最低和最高的最中间，进度条是 50%（对应数字 0.5）
新数值 = （当前数值-最小值）/（最大值-最小值）
它的特点是具有“强边界”，缩放后，数据绝对不可能小于0，也绝对不可能大于1
而标准化的特点是：寻找数据的“重心“（平均值），看看每个数据偏离重心的距离
同样是那场考试，平均分是100分
考 100 分的人，刚好是平均水平，所以他的新数值是 0
考 120 分的人，比平均分高，新数值可能是 +1.5
考 80 分的人，比平均分低，新数值可能是 -1.5

归一化的代码写法跟标准化完全一致

怎么选择
大多数场景选择标准化，但如果当对输入范围有严格要求时，归一化效果更好

正则化(Normalizer)主要是按行缩放。严格来说，是样本缩放(Sample Scaling)，把这一行的全部特征压缩成一个长度为1的向量
代码写法跟前两种完全一致